# Thermodynamics

Thermodynamics governs energy transfer and transformation in physical systems. It connects macroscopic observables — temperature, pressure, volume — to the flow of heat and the performance of work.

## Systems and State Variables

A **thermodynamic system** is the portion of the universe under study, separated from its **surroundings** by a boundary. Systems are classified by what crosses the boundary:

| Type | Energy exchange | Matter exchange |
|:-----|:---------------|:---------------|
| **Open** | Yes | Yes |
| **Closed** | Yes | No |
| **Isolated** | No | No |

The **state** of a system in equilibrium is fully specified by a small set of **state variables** (path-independent):

- $T$ — Temperature (K)
- $P$ — Pressure (Pa)
- $V$ — Volume (m$^3$)
- $U$ — Internal energy (J) — total microscopic kinetic + potential energy
- $S$ — Entropy (J/K) — measure of microscopic disorder
- $H = U + PV$ — Enthalpy (J) — useful for constant-pressure processes

In contrast, **heat** $Q$ and **work** $W$ are **path-dependent** (process quantities, not state functions). We write their infinitesimals as $\delta Q$ and $\delta W$ (inexact differentials) rather than $dQ$, $dW$.

**Equations of state** relate state variables. The simplest is the ideal gas law, which we explore next.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

## Ideal Gas Law

For $n$ moles of an ideal gas at temperature $T$:

$$PV = nRT$$

where $R = 8.314\;\text{J/(mol\,K)}$ is the universal gas constant. Equivalently, for $N$ molecules with Boltzmann constant $k_B = R/N_A$:

$$PV = Nk_BT$$

**Isotherms** are curves of constant $T$ in the $P$-$V$ plane. Rearranging:

$$P = \frac{nRT}{V}$$

Each isotherm is a hyperbola. Higher temperatures push the isotherm outward (higher $P$ at the same $V$). More moles of gas also shift isotherms upward.

> Adjust $T$ and $n$ below to see how isotherms shift. A family of isotherms at different temperatures is shown simultaneously.

In [2]:
def build_ideal_gas_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    T_w = widgets.FloatSlider(value=300, min=100, max=800, step=10,
                              description='T highlighted (K)', style=style, layout=layout)
    n_w = widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1,
                              description='n (mol)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314  # J/(mol K)

    def update(_):
        T_sel, n = T_w.value, n_w.value
        V = np.linspace(0.5e-3, 30e-3, 500)  # m^3

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left: family of isotherms at different T
            temps = np.arange(200, 801, 100)
            cmap = plt.cm.coolwarm
            norm = plt.Normalize(temps.min(), temps.max())
            for T in temps:
                P = n * R * T / V
                lw = 2.5 if abs(T - T_sel) < 1 else 1.0
                alpha = 1.0 if abs(T - T_sel) < 1 else 0.5
                ax1.plot(V * 1e3, P * 1e-3, color=cmap(norm(T)), lw=lw, alpha=alpha)
            # Highlighted isotherm
            P_sel = n * R * T_sel / V
            ax1.plot(V * 1e3, P_sel * 1e-3, color=cmap(norm(T_sel)), lw=3,
                     label=f'T = {T_sel:.0f} K (highlighted)')

            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax1, label='Temperature (K)', pad=0.02)

            ax1.set_xlabel('Volume (L)')
            ax1.set_ylabel('Pressure (kPa)')
            ax1.set_title(f'Ideal Gas Isotherms  |  n = {n:.1f} mol')
            ax1.set_ylim(0, 600)
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            # Right: effect of n at fixed T
            n_vals = [0.5, 1.0, 2.0, 3.0, 5.0]
            colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(n_vals)))
            for ni, c in zip(n_vals, colors):
                P_i = ni * R * T_sel / V
                lw = 2.5 if abs(ni - n) < 0.05 else 1.2
                ax2.plot(V * 1e3, P_i * 1e-3, color=c, lw=lw,
                         label=f'n = {ni:.1f} mol', alpha=1.0 if abs(ni - n) < 0.05 else 0.6)
            ax2.set_xlabel('Volume (L)')
            ax2.set_ylabel('Pressure (kPa)')
            ax2.set_title(f'Effect of n at T = {T_sel:.0f} K')
            ax2.set_ylim(0, 600)
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

    for w in (T_w, n_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([T_w, n_w]), out]))
    update(None)

build_ideal_gas_demo()

## First Law of Thermodynamics

The first law is conservation of energy applied to thermodynamic systems:

$$dU = \delta Q - \delta W$$

- $dU$ — change in internal energy (state function)
- $\delta Q$ — heat added to the system (positive = in)
- $\delta W$ — work done **by** the system (positive = expansion)

For a quasi-static process against pressure $P$:

$$W = \int_{V_1}^{V_2} P\,dV$$

This is the **area under the curve** in the $P$-$V$ diagram. Different paths between the same endpoints yield different amounts of work:

| Process | Constraint | $P(V)$ | Work $W$ |
|:--------|:-----------|:-------|:---------|
| **Isobaric** | $P = \text{const}$ | $P = P_0$ | $P_0(V_2 - V_1)$ |
| **Isothermal** | $T = \text{const}$ | $P = nRT/V$ | $nRT\ln(V_2/V_1)$ |
| **Adiabatic** | $Q = 0$ | $PV^\gamma = \text{const}$ | $\frac{P_1V_1 - P_2V_2}{\gamma - 1}$ |

> Select a process type to see the work (shaded area) on the $P$-$V$ diagram.

In [3]:
def build_first_law_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    process_w = widgets.Dropdown(options=['Isobaric', 'Isothermal', 'Adiabatic'],
                                 value='Isothermal', description='Process type',
                                 style=style, layout=layout)
    V2_w = widgets.FloatSlider(value=15.0, min=6.0, max=25.0, step=0.5,
                               description='V_final (L)', style=style, layout=layout)
    gamma_w = widgets.FloatSlider(value=1.4, min=1.1, max=1.67, step=0.01,
                                  description='gamma (adiabatic)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314
    n = 1.0
    T1 = 300.0
    V1 = 5.0e-3  # 5 L in m^3
    P1 = n * R * T1 / V1

    def update(_):
        proc = process_w.value
        V2 = V2_w.value * 1e-3
        gamma = gamma_w.value
        V = np.linspace(V1, V2, 500)

        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(1, 1, figsize=(13, 5))

            if proc == 'Isobaric':
                P_curve = np.full_like(V, P1)
                W = P1 * (V2 - V1)
                Q = n * (7/2) * R * (V2/V1 - 1) * T1  # diatomic Cp
                dU = Q - W
                color = '#2ca02c'
                label = f'Isobaric (P = {P1*1e-3:.1f} kPa)'
            elif proc == 'Isothermal':
                P_curve = n * R * T1 / V
                W = n * R * T1 * np.log(V2 / V1)
                Q = W  # dU = 0 for isothermal ideal gas
                dU = 0.0
                color = '#1f77b4'
                label = f'Isothermal (T = {T1:.0f} K)'
            else:  # Adiabatic
                P_curve = P1 * (V1 / V) ** gamma
                P2 = P1 * (V1 / V2) ** gamma
                W = (P1 * V1 - P2 * V2) / (gamma - 1)
                Q = 0.0
                dU = -W
                color = '#d62728'
                label = f'Adiabatic (gamma = {gamma:.2f})'

            ax.plot(V * 1e3, P_curve * 1e-3, color=color, lw=2.5, label=label)
            ax.fill_between(V * 1e3, 0, P_curve * 1e-3, alpha=0.2, color=color,
                            label=f'W = {W:.1f} J')
            ax.plot(V1 * 1e3, P1 * 1e-3, 'ko', ms=8, zorder=5)
            ax.annotate(f'  ({V1*1e3:.0f} L, {P1*1e-3:.0f} kPa)',
                        (V1 * 1e3, P1 * 1e-3), fontsize=9)
            P_end = P_curve[-1]
            ax.plot(V2 * 1e3, P_end * 1e-3, 'ko', ms=8, zorder=5)
            ax.annotate(f'  ({V2*1e3:.0f} L, {P_end*1e-3:.0f} kPa)',
                        (V2 * 1e3, P_end * 1e-3), fontsize=9)

            # Info text
            info = (f'W = {W:.1f} J\n'
                    f'Q = {Q:.1f} J\n'
                    f'dU = {dU:.1f} J\n'
                    f'Check: Q - W = {Q - W:.1f} J = dU')
            ax.text(0.98, 0.95, info, transform=ax.transAxes, fontsize=10,
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8))

            ax.set_xlabel('Volume (L)')
            ax.set_ylabel('Pressure (kPa)')
            ax.set_title('First Law: Work = Area Under P-V Curve')
            ax.set_ylim(0, None)
            ax.legend(fontsize=10, loc='upper center')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    for w in (process_w, V2_w, gamma_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([process_w, V2_w, gamma_w]), out]))
    update(None)

build_first_law_demo()

## Thermodynamic Processes

Four fundamental quasi-static processes connect states of an ideal gas. Starting from the same initial state $(P_1, V_1, T_1)$:

### 1. Isothermal ($T = \text{const}$)
$$PV = nRT_1 = \text{const} \quad\Longrightarrow\quad P = \frac{nRT_1}{V}$$

### 2. Isobaric ($P = \text{const}$)
$$P = P_1, \quad V/T = \text{const}$$

### 3. Isochoric ($V = \text{const}$)
$$V = V_1, \quad P/T = \text{const} \quad\Longrightarrow\quad \text{vertical line on }P\text{-}V\text{ diagram}$$

### 4. Adiabatic ($Q = 0$)
$$PV^\gamma = P_1 V_1^\gamma = \text{const} \quad\Longrightarrow\quad P = P_1\left(\frac{V_1}{V}\right)^\gamma$$

where $\gamma = C_P/C_V$ is the heat capacity ratio:

| Gas type | Degrees of freedom | $\gamma$ |
|:---------|:-------------------|:---------|
| Monatomic (He, Ar) | 3 | 5/3 = 1.667 |
| Diatomic (N$_2$, O$_2$) | 5 | 7/5 = 1.400 |
| Polyatomic (CO$_2$) | 6+ | ~1.3 |

The adiabat is always **steeper** than the isotherm through the same point because $\gamma > 1$.

> Adjust $\gamma$ to compare all four processes from the same initial state on a single $P$-$V$ diagram.

In [4]:
def build_processes_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    gamma_w = widgets.FloatSlider(value=1.4, min=1.1, max=1.67, step=0.01,
                                  description='gamma', style=style, layout=layout)
    V1_w = widgets.FloatSlider(value=5.0, min=2.0, max=10.0, step=0.5,
                               description='V_initial (L)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314
    n = 1.0
    T1 = 400.0

    def update(_):
        gamma = gamma_w.value
        V1 = V1_w.value * 1e-3
        P1 = n * R * T1 / V1
        V = np.linspace(V1 * 0.3, V1 * 5, 500)
        V_exp = np.linspace(V1, V1 * 5, 500)  # expansion only

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Isothermal
            P_iso = n * R * T1 / V
            ax1.plot(V * 1e3, P_iso * 1e-3, 'b-', lw=2, label='Isothermal')

            # Isobaric
            ax1.axhline(P1 * 1e-3, color='green', lw=2, ls='-', label='Isobaric')

            # Isochoric
            P_isochoric = np.linspace(P1 * 0.3, P1 * 2, 100)
            ax1.axvline(V1 * 1e3, color='orange', lw=2, ls='-', label='Isochoric')

            # Adiabatic
            P_adi = P1 * (V1 / V) ** gamma
            ax1.plot(V * 1e3, P_adi * 1e-3, 'r-', lw=2, label=f'Adiabatic (gamma={gamma:.2f})')

            # Initial state
            ax1.plot(V1 * 1e3, P1 * 1e-3, 'ko', ms=10, zorder=5, label=f'Initial state')

            ax1.set_xlabel('Volume (L)')
            ax1.set_ylabel('Pressure (kPa)')
            ax1.set_title(f'Four Processes from ({V1*1e3:.1f} L, {P1*1e-3:.0f} kPa, {T1:.0f} K)')
            ax1.set_xlim(0, V1 * 5 * 1e3)
            ax1.set_ylim(0, P1 * 2 * 1e-3)
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            # Right: work comparison (expansion from V1 to 3*V1)
            V2 = V1 * 3
            V_work = np.linspace(V1, V2, 500)

            P_iso_w = n * R * T1 / V_work
            P_adi_w = P1 * (V1 / V_work) ** gamma
            P_isob_w = np.full_like(V_work, P1)

            W_iso = n * R * T1 * np.log(V2 / V1)
            P2_adi = P1 * (V1 / V2) ** gamma
            W_adi = (P1 * V1 - P2_adi * V2) / (gamma - 1)
            W_isob = P1 * (V2 - V1)

            ax2.fill_between(V_work * 1e3, 0, P_isob_w * 1e-3, alpha=0.15, color='green')
            ax2.fill_between(V_work * 1e3, 0, P_iso_w * 1e-3, alpha=0.15, color='blue')
            ax2.fill_between(V_work * 1e3, 0, P_adi_w * 1e-3, alpha=0.15, color='red')

            ax2.plot(V_work * 1e3, P_isob_w * 1e-3, 'g-', lw=2,
                     label=f'Isobaric: W = {W_isob:.0f} J')
            ax2.plot(V_work * 1e3, P_iso_w * 1e-3, 'b-', lw=2,
                     label=f'Isothermal: W = {W_iso:.0f} J')
            ax2.plot(V_work * 1e3, P_adi_w * 1e-3, 'r-', lw=2,
                     label=f'Adiabatic: W = {W_adi:.0f} J')

            ax2.set_xlabel('Volume (L)')
            ax2.set_ylabel('Pressure (kPa)')
            ax2.set_title(f'Work Comparison: V = {V1*1e3:.0f} L to {V2*1e3:.0f} L')
            ax2.set_ylim(0, P1 * 1.5 * 1e-3)
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

    for w in (gamma_w, V1_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([gamma_w, V1_w]), out]))
    update(None)

build_processes_demo()

## Heat Engines & Carnot Cycle

A **heat engine** converts heat into work by cycling a working substance between a hot reservoir ($T_h$) and a cold reservoir ($T_c$). Per cycle:

$$W_{\text{net}} = Q_h - Q_c$$

The **thermal efficiency** is the fraction of input heat converted to work:

$$\eta = \frac{W_{\text{net}}}{Q_h} = 1 - \frac{Q_c}{Q_h}$$

### The Carnot Cycle

The most efficient possible cycle between two temperatures consists of four reversible steps:

1. **Isothermal expansion** at $T_h$: absorb $Q_h$ from hot reservoir
2. **Adiabatic expansion**: temperature drops from $T_h$ to $T_c$
3. **Isothermal compression** at $T_c$: reject $Q_c$ to cold reservoir
4. **Adiabatic compression**: temperature rises from $T_c$ to $T_h$

The **Carnot efficiency** depends only on the reservoir temperatures:

$$\eta_{\text{Carnot}} = 1 - \frac{T_c}{T_h}$$

No real engine can exceed $\eta_{\text{Carnot}}$ (Carnot's theorem).

### Other Ideal Cycles

| Cycle | Application | Process sequence | Efficiency |
|:------|:-----------|:----------------|:-----------|
| **Otto** | Gasoline engine | Two adiabats + two isochoric | $1 - r^{1-\gamma}$ |
| **Diesel** | Diesel engine | Two adiabats + isochoric + isobaric | $1 - \frac{r_c^\gamma - 1}{\gamma(r_c-1)r^{\gamma-1}}$ |

where $r$ is the compression ratio $V_{\max}/V_{\min}$.

> Select a cycle type and adjust the reservoir temperatures to see the $P$-$V$ diagram, efficiency, and energy flows.

In [5]:
def build_engine_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='380px')
    cycle_w = widgets.Dropdown(options=['Carnot', 'Otto', 'Diesel'],
                               value='Carnot', description='Cycle',
                               style=style, layout=layout)
    Th_w = widgets.FloatSlider(value=600, min=400, max=1200, step=10,
                               description='T_hot (K)', style=style, layout=layout)
    Tc_w = widgets.FloatSlider(value=300, min=200, max=500, step=10,
                               description='T_cold (K)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314
    n = 1.0
    gamma = 1.4

    def update(_):
        Th, Tc = Th_w.value, Tc_w.value
        if Tc >= Th:
            Tc = Th - 10

        cycle = cycle_w.value

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

            if cycle == 'Carnot':
                # State 1: start of isothermal expansion at Th
                V1 = 5e-3
                P1 = n * R * Th / V1
                # State 2: end of isothermal expansion at Th
                V2 = V1 * 2.5
                P2 = n * R * Th / V2
                # State 3: end of adiabatic expansion (Th -> Tc)
                V3 = V2 * (Th / Tc) ** (1 / (gamma - 1))
                P3 = n * R * Tc / V3
                # State 4: end of isothermal compression at Tc
                V4 = V1 * (Th / Tc) ** (1 / (gamma - 1))
                P4 = n * R * Tc / V4

                # Path 1->2: isothermal expansion at Th
                V12 = np.linspace(V1, V2, 200)
                P12 = n * R * Th / V12
                # Path 2->3: adiabatic expansion
                V23 = np.linspace(V2, V3, 200)
                P23 = P2 * (V2 / V23) ** gamma
                # Path 3->4: isothermal compression at Tc
                V34 = np.linspace(V3, V4, 200)
                P34 = n * R * Tc / V34
                # Path 4->1: adiabatic compression
                V41 = np.linspace(V4, V1, 200)
                P41 = P4 * (V4 / V41) ** gamma

                ax1.plot(V12 * 1e3, P12 * 1e-3, 'r-', lw=2.5, label=f'Isothermal exp. (Th={Th:.0f} K)')
                ax1.plot(V23 * 1e3, P23 * 1e-3, 'b--', lw=2, label='Adiabatic exp.')
                ax1.plot(V34 * 1e3, P34 * 1e-3, 'c-', lw=2.5, label=f'Isothermal comp. (Tc={Tc:.0f} K)')
                ax1.plot(V41 * 1e3, P41 * 1e-3, 'm--', lw=2, label='Adiabatic comp.')

                # Fill the cycle
                V_cycle = np.concatenate([V12, V23, V34[::-1], V41[::-1]])
                P_cycle = np.concatenate([P12, P23, P34[::-1], P41[::-1]])
                ax1.fill(V_cycle * 1e3, P_cycle * 1e-3, alpha=0.1, color='gold')

                Qh = n * R * Th * np.log(V2 / V1)
                Qc = n * R * Tc * np.log(V3 / V4)
                W_net = Qh - Qc
                eta = 1 - Tc / Th

                # Mark states
                for i, (Vp, Pp, lbl) in enumerate([
                    (V1, P1, '1'), (V2, P2, '2'), (V3, P3, '3'), (V4, P4, '4')]):
                    ax1.plot(Vp * 1e3, Pp * 1e-3, 'ko', ms=7, zorder=5)
                    ax1.annotate(f'  {lbl}', (Vp * 1e3, Pp * 1e-3), fontsize=11, fontweight='bold')

            elif cycle == 'Otto':
                V_min = 2e-3
                r = 8.0  # compression ratio
                V_max = r * V_min

                # State 1: BDC after intake
                T1 = Tc
                P1_state = n * R * T1 / V_max
                # State 2: TDC after adiabatic compression
                T2 = T1 * r ** (gamma - 1)
                P2_state = n * R * T2 / V_min
                # State 3: TDC after isochoric heat addition
                T3 = Th
                P3_state = n * R * T3 / V_min
                # State 4: BDC after adiabatic expansion
                T4 = T3 / r ** (gamma - 1)
                P4_state = n * R * T4 / V_max

                # Paths
                V_12 = np.linspace(V_max, V_min, 200)
                P_12 = P1_state * (V_max / V_12) ** gamma
                V_23 = np.array([V_min, V_min])
                P_23 = np.array([P2_state, P3_state])
                V_34 = np.linspace(V_min, V_max, 200)
                P_34 = P3_state * (V_min / V_34) ** gamma
                V_41 = np.array([V_max, V_max])
                P_41 = np.array([P4_state, P1_state])

                ax1.plot(V_12 * 1e3, P_12 * 1e-3, 'b--', lw=2, label='Adiabatic comp. (1-2)')
                ax1.plot(V_23 * 1e3, P_23 * 1e-3, 'r-', lw=2.5, label='Isochoric heat in (2-3)')
                ax1.plot(V_34 * 1e3, P_34 * 1e-3, 'm--', lw=2, label='Adiabatic exp. (3-4)')
                ax1.plot(V_41 * 1e3, P_41 * 1e-3, 'c-', lw=2.5, label='Isochoric heat out (4-1)')

                V_cycle = np.concatenate([V_12, [V_min], V_34, [V_max]])
                P_cycle = np.concatenate([P_12, [P3_state], P_34, [P1_state]])
                ax1.fill(V_cycle * 1e3, P_cycle * 1e-3, alpha=0.1, color='gold')

                Cv = R * (1 / (gamma - 1))
                Qh = n * Cv * (T3 - T2)
                Qc = n * Cv * (T4 - T1)
                W_net = Qh - Qc
                eta = 1 - r ** (1 - gamma)

                for Vp, Pp, lbl in [(V_max, P1_state, '1'), (V_min, P2_state, '2'),
                                     (V_min, P3_state, '3'), (V_max, P4_state, '4')]:
                    ax1.plot(Vp * 1e3, Pp * 1e-3, 'ko', ms=7, zorder=5)
                    ax1.annotate(f'  {lbl}', (Vp * 1e3, Pp * 1e-3), fontsize=11, fontweight='bold')

            else:  # Diesel
                V_min = 2e-3
                r = 18.0  # compression ratio
                V_max = r * V_min
                rc = 2.5  # cutoff ratio
                V_cutoff = rc * V_min

                T1 = Tc
                P1_state = n * R * T1 / V_max
                T2 = T1 * r ** (gamma - 1)
                P2_state = n * R * T2 / V_min
                T3 = T2 * rc  # isobaric
                P3_state = P2_state  # isobaric
                T4 = T3 * (V_cutoff / V_max) ** (gamma - 1)
                P4_state = n * R * T4 / V_max

                V_12 = np.linspace(V_max, V_min, 200)
                P_12 = P1_state * (V_max / V_12) ** gamma
                V_23 = np.linspace(V_min, V_cutoff, 200)
                P_23 = np.full_like(V_23, P2_state)
                V_34 = np.linspace(V_cutoff, V_max, 200)
                P_34 = P3_state * (V_cutoff / V_34) ** gamma
                V_41 = np.array([V_max, V_max])
                P_41 = np.array([P4_state, P1_state])

                ax1.plot(V_12 * 1e3, P_12 * 1e-3, 'b--', lw=2, label='Adiabatic comp. (1-2)')
                ax1.plot(V_23 * 1e3, P_23 * 1e-3, 'r-', lw=2.5, label='Isobaric heat in (2-3)')
                ax1.plot(V_34 * 1e3, P_34 * 1e-3, 'm--', lw=2, label='Adiabatic exp. (3-4)')
                ax1.plot(V_41 * 1e3, P_41 * 1e-3, 'c-', lw=2.5, label='Isochoric heat out (4-1)')

                V_cycle = np.concatenate([V_12, V_23, V_34, [V_max]])
                P_cycle = np.concatenate([P_12, P_23, P_34, [P1_state]])
                ax1.fill(V_cycle * 1e3, P_cycle * 1e-3, alpha=0.1, color='gold')

                Cp = R * gamma / (gamma - 1)
                Cv = R / (gamma - 1)
                Qh = n * Cp * (T3 - T2)
                Qc = n * Cv * (T4 - T1)
                W_net = Qh - Qc
                eta = 1 - (1 / (gamma * (rc - 1))) * (rc ** gamma - 1) / r ** (gamma - 1)

                for Vp, Pp, lbl in [(V_max, P1_state, '1'), (V_min, P2_state, '2'),
                                     (V_cutoff, P3_state, '3'), (V_max, P4_state, '4')]:
                    ax1.plot(Vp * 1e3, Pp * 1e-3, 'ko', ms=7, zorder=5)
                    ax1.annotate(f'  {lbl}', (Vp * 1e3, Pp * 1e-3), fontsize=11, fontweight='bold')

            ax1.set_xlabel('Volume (L)')
            ax1.set_ylabel('Pressure (kPa)')
            ax1.set_title(f'{cycle} Cycle  |  P-V Diagram')
            ax1.set_ylim(0, None)
            ax1.legend(fontsize=8.5, loc='upper right')
            ax1.grid(True, alpha=0.3)

            # Right: energy bar chart
            bars = ax2.bar(['Q_h (in)', 'Q_c (out)', 'W_net'], [Qh, Qc, W_net],
                           color=['#d62728', '#1f77b4', '#2ca02c'], alpha=0.7, edgecolor='black')
            for bar, val in zip(bars, [Qh, Qc, W_net]):
                ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                         f'{val:.0f} J', ha='center', fontsize=11, fontweight='bold')

            ax2.set_ylabel('Energy (J)')
            ax2.set_title(f'{cycle} Cycle  |  eta = {eta*100:.1f}%')
            ax2.grid(True, alpha=0.3, axis='y')

            info = (f'Efficiency: eta = {eta*100:.1f}%\n'
                    f'Q_h = {Qh:.0f} J\n'
                    f'Q_c = {Qc:.0f} J\n'
                    f'W_net = Q_h - Q_c = {W_net:.0f} J')
            ax2.text(0.98, 0.95, info, transform=ax2.transAxes, fontsize=10,
                     va='top', ha='right',
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

            plt.tight_layout()
            plt.show()

    for w in (cycle_w, Th_w, Tc_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([cycle_w, Th_w, Tc_w]), out]))
    update(None)

build_engine_demo()

## Entropy

**Entropy** $S$ quantifies the microscopic disorder of a system. For a reversible process:

$$dS = \frac{\delta Q_{\text{rev}}}{T}$$

For an ideal gas undergoing a general process from state 1 to state 2:

$$\Delta S = nC_V \ln\frac{T_2}{T_1} + nR\ln\frac{V_2}{V_1}$$

or equivalently:

$$\Delta S = nC_P \ln\frac{T_2}{T_1} - nR\ln\frac{P_2}{P_1}$$

### The Carnot Cycle on the $T$-$S$ Diagram

The Carnot cycle is remarkably simple in the $T$-$S$ plane — it forms a **rectangle**:

- **Isothermal expansion** at $T_h$: horizontal line (entropy increases from $S_1$ to $S_2$)
- **Adiabatic expansion**: vertical line down ($S = \text{const}$, temperature drops)
- **Isothermal compression** at $T_c$: horizontal line (entropy decreases)
- **Adiabatic compression**: vertical line up ($S = \text{const}$, temperature rises)

The **area enclosed** in the $T$-$S$ diagram equals the net work:

$$W_{\text{net}} = \oint T\,dS = (T_h - T_c)(S_2 - S_1)$$

> Adjust $T_h$ and $T_c$ to see the Carnot rectangle and verify that the enclosed area equals the net work.

In [6]:
def build_entropy_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    Th_w = widgets.FloatSlider(value=600, min=400, max=1200, step=10,
                               description='T_hot (K)', style=style, layout=layout)
    Tc_w = widgets.FloatSlider(value=300, min=200, max=500, step=10,
                               description='T_cold (K)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314
    n = 1.0

    def update(_):
        Th, Tc = Th_w.value, Tc_w.value
        if Tc >= Th:
            Tc = Th - 10

        # Carnot cycle entropy change during isothermal expansion
        V1, V2 = 5e-3, 12.5e-3  # expansion ratio
        S1 = 0.0  # reference
        dS_iso = n * R * np.log(V2 / V1)
        S2 = S1 + dS_iso

        Qh = Th * dS_iso
        Qc = Tc * dS_iso
        W_net = (Th - Tc) * dS_iso
        eta = 1 - Tc / Th

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # T-S diagram (rectangle)
            S_rect = [S1, S2, S2, S1, S1]
            T_rect = [Th, Th, Tc, Tc, Th]
            ax1.fill(S_rect, T_rect, alpha=0.2, color='gold', label=f'W_net = {W_net:.0f} J')
            ax1.plot(S_rect, T_rect, 'k-', lw=2)

            # Label each process
            ax1.annotate('', xy=(S2, Th), xytext=(S1, Th),
                         arrowprops=dict(arrowstyle='->', color='red', lw=2))
            ax1.text((S1 + S2) / 2, Th + 15, f'Isothermal exp. (Q_h = {Qh:.0f} J)',
                     ha='center', color='red', fontsize=9)

            ax1.annotate('', xy=(S2, Tc), xytext=(S2, Th),
                         arrowprops=dict(arrowstyle='->', color='blue', lw=2))
            ax1.text(S2 + 0.3, (Th + Tc) / 2, 'Adiabatic exp.',
                     ha='left', color='blue', fontsize=9, rotation=-90, va='center')

            ax1.annotate('', xy=(S1, Tc), xytext=(S2, Tc),
                         arrowprops=dict(arrowstyle='->', color='cyan', lw=2))
            ax1.text((S1 + S2) / 2, Tc - 30, f'Isothermal comp. (Q_c = {Qc:.0f} J)',
                     ha='center', color='teal', fontsize=9)

            ax1.annotate('', xy=(S1, Th), xytext=(S1, Tc),
                         arrowprops=dict(arrowstyle='->', color='purple', lw=2))
            ax1.text(S1 - 0.3, (Th + Tc) / 2, 'Adiabatic comp.',
                     ha='right', color='purple', fontsize=9, rotation=90, va='center')

            # Mark corners
            for Si, Ti, lbl in [(S1, Th, '1'), (S2, Th, '2'), (S2, Tc, '3'), (S1, Tc, '4')]:
                ax1.plot(Si, Ti, 'ko', ms=7, zorder=5)
                ax1.annotate(f'  {lbl}', (Si, Ti), fontsize=11, fontweight='bold')

            ax1.set_xlabel('Entropy S (J/K)')
            ax1.set_ylabel('Temperature T (K)')
            ax1.set_title('Carnot Cycle on T-S Diagram')
            ax1.set_ylim(100, Th + 100)
            ax1.legend(fontsize=10)
            ax1.grid(True, alpha=0.3)

            # Right: energy flow diagram
            labels = ['Q_h (absorbed)', 'W_net (output)', 'Q_c (rejected)']
            values = [Qh, W_net, Qc]
            colors = ['#d62728', '#2ca02c', '#1f77b4']
            bars = ax2.barh(labels, values, color=colors, alpha=0.7, edgecolor='black')
            for bar, val in zip(bars, values):
                ax2.text(val + 10, bar.get_y() + bar.get_height() / 2,
                         f'{val:.0f} J', va='center', fontsize=11, fontweight='bold')

            ax2.set_xlabel('Energy (J)')
            ax2.set_title(f'Carnot Efficiency = {eta*100:.1f}%')
            ax2.grid(True, alpha=0.3, axis='x')

            info = (f'T_h = {Th:.0f} K, T_c = {Tc:.0f} K\n'
                    f'eta = 1 - T_c/T_h = {eta*100:.1f}%\n'
                    f'dS = {dS_iso:.2f} J/K\n'
                    f'Area = (T_h - T_c) * dS = {W_net:.0f} J')
            ax2.text(0.98, 0.05, info, transform=ax2.transAxes, fontsize=10,
                     va='bottom', ha='right',
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

            plt.tight_layout()
            plt.show()

    for w in (Th_w, Tc_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([Th_w, Tc_w]), out]))
    update(None)

build_entropy_demo()

## Second Law & Entropy

The **Second Law of Thermodynamics** can be stated in several equivalent ways:

- **Clausius statement**: Heat cannot spontaneously flow from cold to hot.
- **Kelvin-Planck statement**: No engine can convert heat entirely into work.
- **Entropy statement**: The total entropy of an isolated system can only increase or stay the same:

$$\Delta S_{\text{total}} = \Delta S_{\text{system}} + \Delta S_{\text{surroundings}} \geq 0$$

### Clausius Inequality

For any cyclic process:

$$\oint \frac{\delta Q}{T} \leq 0$$

Equality holds only for reversible cycles.

### Free Expansion

A classic irreversible process: an ideal gas expands into a vacuum from volume $V_1$ to $V_2$.

- $W = 0$ (expansion against zero pressure)
- $Q = 0$ (adiabatic walls)
- $\Delta U = 0 \Rightarrow \Delta T = 0$ (ideal gas)

Yet entropy **increases**:

$$\Delta S = nR\ln\frac{V_2}{V_1} > 0$$

This entropy increase reflects the irreversibility — the gas will never spontaneously re-compress.

> Adjust the volume ratio $V_2/V_1$ to see the entropy change during free expansion.

In [ ]:
def build_second_law_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    ratio_w = widgets.FloatSlider(value=3.0, min=1.1, max=10.0, step=0.1,
                                  description='V2/V1 (expansion ratio)', style=style, layout=layout)
    n_w = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1,
                              description='n (mol)', style=style, layout=layout)
    out = widgets.Output()

    R = 8.314

    def update(_):
        ratio = ratio_w.value
        n = n_w.value
        dS = n * R * np.log(ratio)

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left: entropy change vs volume ratio
            ratios = np.linspace(1.0, 10.0, 300)
            dS_curve = n * R * np.log(ratios)
            ax1.plot(ratios, dS_curve, 'steelblue', lw=2, label=f'dS = nR ln(V2/V1)')
            ax1.fill_between(ratios, 0, dS_curve, alpha=0.15, color='steelblue')
            ax1.axvline(ratio, color='#d62728', ls='--', lw=2,
                        label=f'V2/V1 = {ratio:.1f}, dS = {dS:.2f} J/K')
            ax1.plot(ratio, dS, 'ro', ms=10, zorder=5)
            ax1.axhline(0, color='black', lw=0.8)
            ax1.set_xlabel('Volume ratio V2/V1')
            ax1.set_ylabel('Entropy change dS (J/K)')
            ax1.set_title(f'Free Expansion: n = {n:.1f} mol')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            # Right: comparison of reversible vs irreversible paths
            V1 = 5.0  # L (arbitrary reference)
            V2 = V1 * ratio
            T = 300  # K
            V_path = np.linspace(V1, V2, 300)

            # Reversible isothermal: P = nRT/V
            P_rev = n * R * T / (V_path * 1e-3) * 1e-3  # kPa

            ax2.plot(V_path, P_rev, 'steelblue', lw=2, label='Reversible isothermal path')
            ax2.fill_between(V_path, 0, P_rev, alpha=0.15, color='steelblue',
                             label=f'W_rev = {n * R * T * np.log(ratio):.0f} J')

            # Free expansion: P_ext = 0 (no work)
            ax2.plot([V1, V1, V2], [n * R * T / (V1 * 1e-3) * 1e-3, 0, 0],
                     'r--', lw=2.5, label='Free expansion (W = 0)')
            ax2.plot(V2, n * R * T / (V2 * 1e-3) * 1e-3, 'ro', ms=10, zorder=5)

            ax2.set_xlabel('Volume (L)')
            ax2.set_ylabel('External Pressure (kPa)')
            ax2.set_title('Reversible vs Irreversible Paths (same dS)')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(0, None)

            info = (f'Free expansion (irreversible):\n'
                    f'  W = 0, Q = 0, dU = 0, dT = 0\n'
                    f'  dS_system = nR ln(V2/V1) = {dS:.2f} J/K\n'
                    f'  dS_surr = 0 (no heat exchange)\n'
                    f'  dS_total = {dS:.2f} J/K > 0 (irreversible)')
            ax2.text(0.98, 0.95, info, transform=ax2.transAxes, fontsize=9,
                     va='top', ha='right',
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='mistyrose', alpha=0.9))

            plt.tight_layout()
            plt.show()

    for w in (ratio_w, n_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([ratio_w, n_w]), out]))
    update(None)

build_second_law_demo()

## Carnot Efficiency & Real Engines

The Carnot efficiency $\eta = 1 - T_c/T_h$ sets an **absolute upper bound** on any heat engine operating between temperatures $T_h$ and $T_c$. Real engines fall below this limit due to friction, irreversible heat transfer, and other losses.

Typical efficiencies in practice:

| Engine type | Operating temperatures | Typical $\eta$ | Carnot limit |
|:-----------|:---------------------|:-------------|:-------------|
| Gasoline car | $T_h \approx 2500$ K, $T_c \approx 300$ K | ~25% | ~88% |
| Diesel engine | $T_h \approx 2000$ K, $T_c \approx 300$ K | ~35% | ~85% |
| Coal power plant | $T_h \approx 800$ K, $T_c \approx 300$ K | ~35-40% | ~63% |
| Nuclear power plant | $T_h \approx 570$ K, $T_c \approx 300$ K | ~33% | ~47% |
| Combined-cycle gas turbine | $T_h \approx 1500$ K, $T_c \approx 300$ K | ~60% | ~80% |

> Explore how the Carnot efficiency depends on the temperature ratio, and see where real engines sit relative to the theoretical limit.

In [ ]:
def build_carnot_efficiency_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    Th_w = widgets.FloatSlider(value=800, min=350, max=2500, step=10,
                               description='T_hot (K)', style=style, layout=layout)
    Tc_w = widgets.FloatSlider(value=300, min=200, max=500, step=10,
                               description='T_cold (K)', style=style, layout=layout)
    out = widgets.Output()

    # Real engine data: (name, Th, Tc, actual_eta)
    engines = [
        ('Gasoline car', 2500, 300, 0.25),
        ('Diesel engine', 2000, 300, 0.35),
        ('Coal plant', 800, 300, 0.37),
        ('Nuclear plant', 570, 300, 0.33),
        ('Combined-cycle', 1500, 300, 0.60),
    ]

    def update(_):
        Th, Tc = Th_w.value, Tc_w.value
        if Tc >= Th:
            Tc = Th - 10

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left: Carnot efficiency vs Tc/Th
            ratio = np.linspace(0.01, 0.99, 300)
            eta_carnot = 1 - ratio
            ax1.plot(ratio, eta_carnot * 100, 'k-', lw=2.5, label='Carnot limit')
            ax1.fill_between(ratio, 0, eta_carnot * 100, alpha=0.08, color='green',
                             label='Theoretically accessible')
            ax1.fill_between(ratio, eta_carnot * 100, 100, alpha=0.08, color='red',
                             label='Forbidden by 2nd Law')

            # Mark current selection
            r_sel = Tc / Th
            eta_sel = (1 - r_sel) * 100
            ax1.plot(r_sel, eta_sel, 'ro', ms=12, zorder=5,
                     label=f'Selected: eta = {eta_sel:.1f}%')

            # Mark real engines
            markers = ['s', 'D', '^', 'v', 'p']
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#d62728']
            for (name, Th_e, Tc_e, eta_real), mk, c in zip(engines, markers, colors):
                r_e = Tc_e / Th_e
                eta_c = (1 - r_e) * 100
                ax1.plot(r_e, eta_real * 100, marker=mk, color=c, ms=9, zorder=5,
                         label=f'{name} ({eta_real*100:.0f}%)')
                # Draw line from actual to Carnot limit
                ax1.plot([r_e, r_e], [eta_real * 100, eta_c], color=c,
                         ls=':', lw=1.2, alpha=0.6)

            ax1.set_xlabel('Temperature ratio T_c / T_h')
            ax1.set_ylabel('Efficiency (%)')
            ax1.set_title('Carnot Efficiency vs Temperature Ratio')
            ax1.set_xlim(0, 1)
            ax1.set_ylim(0, 100)
            ax1.legend(fontsize=8, loc='upper right')
            ax1.grid(True, alpha=0.3)

            # Right: efficiency vs Th at fixed Tc
            Th_range = np.linspace(Tc + 10, 3000, 300)
            eta_vs_Th = (1 - Tc / Th_range) * 100
            ax2.plot(Th_range, eta_vs_Th, 'steelblue', lw=2.5,
                     label=f'Carnot limit (T_c = {Tc:.0f} K)')
            ax2.fill_between(Th_range, 0, eta_vs_Th, alpha=0.1, color='steelblue')
            ax2.axvline(Th, color='red', ls='--', lw=2)
            ax2.plot(Th, eta_sel, 'ro', ms=12, zorder=5,
                     label=f'T_h = {Th:.0f} K -> eta = {eta_sel:.1f}%')

            # Mark real engines on this plot too
            for (name, Th_e, Tc_e, eta_real), mk, c in zip(engines, markers, colors):
                ax2.plot(Th_e, eta_real * 100, marker=mk, color=c, ms=9, zorder=5,
                         label=f'{name}')

            ax2.set_xlabel('T_hot (K)')
            ax2.set_ylabel('Efficiency (%)')
            ax2.set_title(f'Efficiency vs T_hot  (T_c = {Tc:.0f} K fixed)')
            ax2.set_ylim(0, 100)
            ax2.legend(fontsize=8, loc='lower right')
            ax2.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

    for w in (Th_w, Tc_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([Th_w, Tc_w]), out]))
    update(None)

build_carnot_efficiency_demo()

## Heat Transfer Modes

Heat transfers from hotter to colder regions via three mechanisms:

### 1. Conduction (Fourier's Law)

Heat flux through a solid is proportional to the temperature gradient:

$$\dot{Q} = -kA\frac{dT}{dx}$$

For a flat wall of thickness $L$ with steady-state temperatures $T_1$ and $T_2$:

$$\dot{Q} = \frac{kA(T_1 - T_2)}{L}$$

| Material | Thermal conductivity $k$ (W/(m K)) |
|:---------|:----------------------------------|
| Copper | 400 |
| Aluminum | 237 |
| Steel | 50 |
| Glass | 1.0 |
| Wood | 0.15 |
| Styrofoam | 0.033 |

### 2. Convection

Heat transfer via fluid motion. Newton's law of cooling:

$$\dot{Q} = hA(T_s - T_\infty)$$

where $h$ is the convection coefficient (W/(m$^2$ K)), typically 5-25 (natural) or 25-250 (forced).

### 3. Radiation (Stefan-Boltzmann Law)

All bodies emit thermal radiation:

$$\dot{Q} = \varepsilon \sigma A T^4$$

where $\sigma = 5.67 \times 10^{-8}$ W/(m$^2$ K$^4$) is the Stefan-Boltzmann constant and $\varepsilon$ is the emissivity (0 to 1).

> Left: temperature profile through a wall with adjustable thermal conductivity and thickness. Right: radiated power vs temperature.

In [ ]:
def build_heat_transfer_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='380px')
    k_w = widgets.FloatLogSlider(value=1.0, base=10, min=-1.5, max=2.7, step=0.1,
                                  description='k (W/(m K))', style=style, layout=layout)
    L_w = widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01,
                              description='Wall thickness L (m)', style=style, layout=layout)
    T1_w = widgets.FloatSlider(value=400, min=300, max=1000, step=10,
                               description='T_inside (K)', style=style, layout=layout)
    T2_w = widgets.FloatSlider(value=280, min=200, max=350, step=5,
                               description='T_outside (K)', style=style, layout=layout)
    out = widgets.Output()

    sigma = 5.67e-8
    A = 1.0  # m^2

    def update(_):
        k = k_w.value
        L = L_w.value
        T1 = T1_w.value
        T2 = T2_w.value

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left: temperature profile through wall
            x = np.linspace(0, L, 200)
            T_profile = T1 + (T2 - T1) * x / L
            Q_dot = k * A * (T1 - T2) / L

            # Show wall region
            ax1.axvspan(0, L, alpha=0.15, color='gray', label='Wall')
            ax1.plot(x, T_profile, 'r-', lw=2.5, label=f'T(x) through wall')

            # Temperature on each side
            ax1.axhline(T1, color='red', ls=':', lw=1.5, alpha=0.5)
            ax1.axhline(T2, color='blue', ls=':', lw=1.5, alpha=0.5)
            ax1.plot(0, T1, 'ro', ms=10, zorder=5)
            ax1.plot(L, T2, 'bo', ms=10, zorder=5)

            # Extend beyond wall
            margin = L * 0.2
            ax1.plot([-margin, 0], [T1, T1], 'r--', lw=1.5, alpha=0.5)
            ax1.plot([L, L + margin], [T2, T2], 'b--', lw=1.5, alpha=0.5)

            ax1.set_xlabel('Position x (m)')
            ax1.set_ylabel('Temperature (K)')
            ax1.set_title(f'Conduction through wall  |  k = {k:.2f} W/(m K)')
            ax1.set_xlim(-margin, L + margin)
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            info = (f'k = {k:.3f} W/(m K)\n'
                    f'L = {L:.2f} m\n'
                    f'dT = {T1 - T2:.0f} K\n'
                    f'Q_dot = k A dT/L\n'
                    f'     = {Q_dot:.1f} W/m^2')
            ax1.text(0.98, 0.5, info, transform=ax1.transAxes, fontsize=9,
                     va='center', ha='right',
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

            # Right: radiation power vs temperature
            T_range = np.linspace(200, 2000, 300)
            for eps, label, c in [(1.0, 'Blackbody (eps=1)', 'black'),
                                   (0.9, 'Oxidized steel (eps=0.9)', '#d62728'),
                                   (0.5, 'Rough surface (eps=0.5)', '#ff7f0e'),
                                   (0.1, 'Polished metal (eps=0.1)', '#2ca02c')]:
                P_rad = eps * sigma * T_range ** 4
                ax2.plot(T_range, P_rad * 1e-3, lw=2, label=label, color=c)

            # Mark current T1
            P_bb = sigma * T1 ** 4
            ax2.axvline(T1, color='gray', ls='--', lw=1.5, alpha=0.5)
            ax2.plot(T1, P_bb * 1e-3, 'ko', ms=8, zorder=5)
            ax2.annotate(f'  T = {T1:.0f} K\n  P = {P_bb*1e-3:.1f} kW/m^2',
                         (T1, P_bb * 1e-3), fontsize=9)

            ax2.set_xlabel('Temperature (K)')
            ax2.set_ylabel('Radiated power (kW/m^2)')
            ax2.set_title('Stefan-Boltzmann Radiation')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(0, None)

            plt.tight_layout()
            plt.show()

    for w in (k_w, L_w, T1_w, T2_w):
        w.observe(update, names='value')
    display(widgets.VBox([widgets.HBox([k_w, L_w]), widgets.HBox([T1_w, T2_w]), out]))
    update(None)

build_heat_transfer_demo()

## Maxwell-Boltzmann Speed Distribution

In an ideal gas at temperature $T$, molecular speeds follow the **Maxwell-Boltzmann distribution**:

$$f(v) = 4\pi n \left(\frac{m}{2\pi k_B T}\right)^{3/2} v^2 \exp\left(-\frac{mv^2}{2k_BT}\right)$$

where $m$ is the molecular mass and $k_B = 1.381 \times 10^{-23}$ J/K is the Boltzmann constant.

Three characteristic speeds:

| Speed | Formula | Physical meaning |
|:------|:--------|:-----------------|
| Most probable $v_{\text{mp}}$ | $\sqrt{\frac{2k_BT}{m}}$ | Peak of $f(v)$ |
| Mean $\langle v \rangle$ | $\sqrt{\frac{8k_BT}{\pi m}}$ | Average speed |
| RMS $v_{\text{rms}}$ | $\sqrt{\frac{3k_BT}{m}}$ | Connects to kinetic energy: $\frac{1}{2}mv_{\text{rms}}^2 = \frac{3}{2}k_BT$ |

Note the ordering: $v_{\text{mp}} < \langle v \rangle < v_{\text{rms}}$ always.

Key observations:
- **Higher $T$**: distribution broadens and shifts right (faster molecules)
- **Lighter molecules**: move faster at the same temperature
- The distribution is **asymmetric** — the high-speed tail extends further than the low-speed side

> Adjust the temperature and select which gases to overlay. The vertical dashed lines mark $v_{\text{mp}}$, $\langle v \rangle$, and $v_{\text{rms}}$ for each gas.

In [ ]:
def build_maxwell_boltzmann_demo():
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='400px')
    T_w = widgets.FloatSlider(value=300, min=100, max=1500, step=10,
                              description='Temperature T (K)', style=style, layout=layout)

    # Gas data: (name, molar mass in kg/mol)
    gas_data = {
        'H2': 2.016e-3,
        'He': 4.003e-3,
        'N2': 28.014e-3,
        'O2': 31.998e-3,
        'CO2': 44.01e-3,
        'Xe': 131.29e-3,
    }

    gas_checkboxes = {name: widgets.Checkbox(value=(name in ['H2', 'N2', 'O2']),
                                             description=name,
                                             layout=widgets.Layout(width='80px'))
                      for name in gas_data}
    out = widgets.Output()

    kB = 1.381e-23  # J/K
    NA = 6.022e23

    def maxwell_boltzmann(v, T, M):
        """Maxwell-Boltzmann speed distribution. M = molar mass (kg/mol)"""
        m = M / NA  # mass per molecule
        coeff = 4 * np.pi * (m / (2 * np.pi * kB * T)) ** 1.5
        return coeff * v ** 2 * np.exp(-m * v ** 2 / (2 * kB * T))

    def update(_):
        T = T_w.value
        selected = [name for name, cb in gas_checkboxes.items() if cb.value]
        if not selected:
            selected = ['N2']

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

            colors = {'H2': '#d62728', 'He': '#ff7f0e', 'N2': '#1f77b4',
                       'O2': '#2ca02c', 'CO2': '#9467bd', 'Xe': '#8c564b'}

            # Left: speed distributions for selected gases at current T
            v_max = 3500 if T > 500 else 2500
            v = np.linspace(0, v_max, 1000)

            for name in selected:
                M = gas_data[name]
                m = M / NA
                f_v = maxwell_boltzmann(v, T, M)
                c = colors[name]

                v_mp = np.sqrt(2 * kB * T / m)
                v_avg = np.sqrt(8 * kB * T / (np.pi * m))
                v_rms = np.sqrt(3 * kB * T / m)

                ax1.plot(v, f_v * 1e3, color=c, lw=2, label=f'{name} (M={M*1e3:.1f} g/mol)')
                ax1.fill_between(v, f_v * 1e3, alpha=0.1, color=c)

                # Mark characteristic speeds
                f_peak = maxwell_boltzmann(v_mp, T, M) * 1e3
                ax1.axvline(v_mp, color=c, ls='--', lw=1, alpha=0.6)
                ax1.axvline(v_avg, color=c, ls='-.', lw=1, alpha=0.6)
                ax1.axvline(v_rms, color=c, ls=':', lw=1.5, alpha=0.6)

                # Small annotations near top
                y_offset = f_peak * 0.95
                ax1.text(v_mp, y_offset, 'mp', color=c, fontsize=7, ha='center',
                         fontweight='bold', alpha=0.7)

            ax1.set_xlabel('Speed v (m/s)')
            ax1.set_ylabel('f(v) (10^-3 s/m)')
            ax1.set_title(f'Maxwell-Boltzmann Distribution  |  T = {T:.0f} K')
            ax1.set_xlim(0, v_max)
            ax1.set_ylim(0, None)
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            # Custom legend for line styles
            from matplotlib.lines import Line2D
            speed_legend = [
                Line2D([0], [0], color='gray', ls='--', lw=1, label='v_mp'),
                Line2D([0], [0], color='gray', ls='-.', lw=1, label='v_avg'),
                Line2D([0], [0], color='gray', ls=':', lw=1.5, label='v_rms'),
            ]
            ax1.legend(handles=ax1.get_legend_handles_labels()[0] + speed_legend,
                       fontsize=8, loc='upper right')

            # Right: distribution of one gas at multiple temperatures
            ref_gas = selected[0]
            M_ref = gas_data[ref_gas]
            temps = [100, 300, 600, 1000, 1500]
            cmap = plt.cm.coolwarm
            norm = plt.Normalize(100, 1500)

            for Ti in temps:
                f_v_i = maxwell_boltzmann(v, Ti, M_ref)
                lw = 2.5 if abs(Ti - T) < 5 else 1.2
                alpha = 1.0 if abs(Ti - T) < 5 else 0.5
                ax2.plot(v, f_v_i * 1e3, color=cmap(norm(Ti)), lw=lw, alpha=alpha,
                         label=f'T = {Ti} K')

            # Current T if not in the preset list
            if not any(abs(Ti - T) < 5 for Ti in temps):
                f_v_curr = maxwell_boltzmann(v, T, M_ref)
                ax2.plot(v, f_v_curr * 1e3, color=cmap(norm(T)), lw=2.5,
                         label=f'T = {T:.0f} K (selected)', zorder=5)

            ax2.set_xlabel('Speed v (m/s)')
            ax2.set_ylabel('f(v) (10^-3 s/m)')
            ax2.set_title(f'{ref_gas}: Effect of Temperature on Speed Distribution')
            ax2.set_xlim(0, v_max)
            ax2.set_ylim(0, None)
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            # Speed table
            table_text = 'Gas    v_mp     v_avg    v_rms\n'
            table_text += '-' * 40 + '\n'
            for name in selected:
                m_i = gas_data[name] / NA
                vmp = np.sqrt(2 * kB * T / m_i)
                vavg = np.sqrt(8 * kB * T / (np.pi * m_i))
                vrms = np.sqrt(3 * kB * T / m_i)
                table_text += f'{name:6s} {vmp:7.0f}  {vavg:7.0f}  {vrms:7.0f} m/s\n'

            ax2.text(0.98, 0.95, table_text, transform=ax2.transAxes, fontsize=8.5,
                     va='top', ha='right', family='monospace',
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

            plt.tight_layout()
            plt.show()

    for w in [T_w] + list(gas_checkboxes.values()):
        w.observe(update, names='value')

    checkbox_row = widgets.HBox(list(gas_checkboxes.values()))
    display(widgets.VBox([T_w, checkbox_row, out]))
    update(None)

build_maxwell_boltzmann_demo()